# AEF-BNG: Local Workflow Example

***Assumes you've ran the example CLI code for London.***

```bash
aef-bng process \
    --year 2025 \
    --bounds 521722 171089 540290 187123 \
    --output ./london_aef
```

This notebook demonstrates working with the GeoParquet output from `aef-bng`:

1. Reading the output files
2. Inspecting the data
3. Dequantising embeddings to analysis-ready float values
4. Visualising embedding bands as an RGB composite

## 0. Read example and plot

Looking at Vauxhall Brdige in London.

In [ ]:
import contextily
import geopandas as gpd

from aef_bng.types import BoundingBox

bbox = (
    BoundingBox(-14824.0062, 6707404.1711, -13687.0054, 6708469.5122)
    .reproject(3857, 27700)
    .as_tuple()
)

gdf = gpd.read_parquet("../london_aef/2025", bbox=bbox)
print(f"Rows:     {len(gdf):,}")

ax = gdf.plot(fc="None", lw=0.4)

contextily.add_basemap(ax=ax, crs=27700)

## 1. Read the full extacted dataset

The pipeline writes GeoParquet files with 10m BNG polygon geometry, a `bng_ref` column, `year`, and 64 int8 embedding bands (`A00`..`A63`).

In [6]:
import geopandas as gpd

gdf = gpd.read_parquet("../london_aef/2025")
print(f"Rows:     {len(gdf):,}")
print(f"CRS:      {gdf.crs.to_epsg()}")
print(f"Geom type: {gdf.geometry.geom_type.iloc[0]}")
print(f"Columns:  {list(gdf.columns[:5])} ... {list(gdf.columns[-3:])}")
gdf.head(3)

Rows:     6,000,000
CRS:      27700
Geom type: Polygon
Columns:  ['bng_ref', 'year', 'A00', 'A01', 'A02'] ... ['A62', 'A63', 'geometry']


,bng_ref,year,A00,A01,A02,A03,A04,A05,A06,A07,...,A55,A56,A57,A58,A59,A60,A61,A62,A63,geometry
0,TQ20007000,2025,-41,-63,-41,16,30,-49,51,-16,...,-51,8,34,-12,48,49,-39,-52,17,"POLYGON ((520010 170000, 520010 170010, 520000..."
1,TQ20017000,2025,-42,-64,-45,20,31,-53,47,-19,...,-49,5,32,-21,46,50,-40,-51,18,"POLYGON ((520020 170000, 520020 170010, 520010..."
2,TQ20027000,2025,-39,-64,-53,28,28,-62,48,-24,...,-47,15,34,-31,38,48,-48,-59,29,"POLYGON ((520030 170000, 520030 170010, 520020..."


In [7]:
# Each geometry is a 10m x 10m BNG cell polygon
cell = gdf.geometry.iloc[0]
print(f"Cell bounds: {cell.bounds}")
print(f"Cell area:   {cell.area} m²  (expected: 100)")

Cell bounds: (520000.0, 170000.0, 520010.0, 170010.0)
Cell area:   100.0 m²  (expected: 100)


In [8]:
# Verify BNG references match geometry
from osbng import BNGReference

sample = gdf.iloc[0]
ref = BNGReference(sample.bng_ref)
ref_poly = ref.bng_to_grid_geom()

print(f"bng_ref:        {sample.bng_ref}")
print(f"Geometry bounds: {sample.geometry.bounds}")
print(f"osbng bounds:    {ref_poly.bounds}")
print(f"Geometry match:           {sample.geometry.equals(ref_poly)}")

bng_ref:        TQ20007000
Geometry bounds: (520000.0, 170000.0, 520010.0, 170010.0)
osbng bounds:    (520000.0, 170000.0, 520010.0, 170010.0)
Geometry match:           True


## 2. Dequantise embeddings

Raw AEF embeddings are stored as int8 values in [-127, 127] (with -128 as nodata).
The dequantisation formula maps these to float32 in [-1, 1]:

```
dequantised = (value / 127.5)² × sign(value)
```

The `dequantise_dataframe` function applies this to all 64 band columns at once.

In [ ]:
from aef_bng.dequantise import dequantise_dataframe

# Dequantise all band columns (A00..A63) from int8 to float32
gdf_dq = dequantise_dataframe(gdf)

print("Before dequantisation:")
print(f"  A00 dtype: {gdf['A00'].dtype}, range: [{gdf['A00'].min()}, {gdf['A00'].max()}]")
print()
print("After dequantisation:")
print(
    f"  A00 dtype: {gdf_dq['A00'].dtype}, range: [{gdf_dq['A00'].min():.4f}, {gdf_dq['A00'].max():.4f}]"  # noqa: E501
)

gdf_dq[["bng_ref", "year", "A00", "A01", "A02"]].head()

Before dequantisation:
  A00 dtype: int8, range: [-74, 57]

After dequantisation:
  A00 dtype: float32, range: [-0.3369, 0.1999]


,bng_ref,year,A00,A01,A02
0,TQ20007000,2025,-0.103406,-0.244152,-0.103406
1,TQ20017000,2025,-0.108512,-0.251965,-0.124567
2,TQ20027000,2025,-0.093564,-0.251965,-0.172795
3,TQ20007001,2025,-0.098424,-0.244152,-0.098424
4,TQ20017001,2025,-0.108512,-0.251965,-0.124567


## 3. RGB visualisation from embedding bands

AEF embeddings encode spectral and spatial features into 64 learned dimensions.
While they don't correspond to specific wavelengths, assigning three bands to
RGB channels gives a useful false-colour visualisation of the landscape.

Different band combinations highlight different landscape features.

In [10]:
import matplotlib.pyplot as plt
import numpy as np


def plot_embedding_rgb(
    gdf,
    r_band="A00",
    g_band="A01",
    b_band="A02",
    title=None,
    figsize=(12, 10),
):
    """Plot an RGB composite from three AEF embedding bands.

    Renders the actual 10m BNG polygon geometry, coloured by normalising
    the selected bands to [0, 1] and mapping to RGB.
    """
    # Extract band values
    r = gdf[r_band].to_numpy().astype(np.float64)
    g = gdf[g_band].to_numpy().astype(np.float64)
    b = gdf[b_band].to_numpy().astype(np.float64)

    # Normalise each channel to [0, 1] with 2-98 percentile stretch
    def norm(arr):
        lo, hi = np.nanpercentile(arr, [2, 98])
        return np.clip((arr - lo) / (hi - lo + 1e-10), 0, 1)

    rgb = np.stack([norm(r), norm(g), norm(b)], axis=-1)

    # Build a per-polygon colour list and plot with geopandas
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    gdf.plot(ax=ax, color=[tuple(c) for c in rgb], linewidth=0, antialiased=True)
    ax.set_axis_off()
    ax.set_aspect("equal")
    ax.set_title(title or f"AEF Embedding RGB: R={r_band}, G={g_band}, B={b_band}", fontsize=18)
    plt.tight_layout()
    return fig, ax

Visualise for a sample area using the first 3 bands.

*Note*: the full dataset will take a long time to plot (~20 minutes).

In [ ]:
bbox[0]

527768.2312700776

In [ ]:
year = gdf_dq.iloc[0].year

selected_columns = ["A00", "A01", "A02"]

bbox = (
    BoundingBox(-17981.8109, 6706393.7692, -7395.2825, 6715489.7755)
    .reproject(3857, 27700)
    .as_tuple()
)

fig, ax = plot_embedding_rgb(
    gdf_dq.cx[bbox[0] : bbox[2], bbox[1] : bbox[3]],
    r_band=selected_columns[0],
    g_band=selected_columns[1],
    b_band=selected_columns[2],
    figsize=(14, 14),
)
plt.show()

View a random selection of bands.

In [ ]:
import random


def get_random_a_columns(num_columns=3):
    """
    Selects a specified number of unique random columns
    matching the pattern A00 to A63.
    """
    random_numbers = random.sample(range(64), num_columns)

    column_names = [f"A{num:02d}" for num in random_numbers]

    return column_names


selected_columns = get_random_a_columns(3)
print(f"Selected columns: {selected_columns}")

fig, ax = plot_embedding_rgb(
    gdf_dq.cx[bbox[0] : bbox[2], bbox[1] : bbox[3]],
    r_band=selected_columns[0],
    g_band=selected_columns[1],
    b_band=selected_columns[2],
    figsize=(14, 14),
)
plt.show()